# 📚 Section 11 (Part 2/2): Business KPI Calculations — Core Business KPIs
> 실무 KPI 계산 (2/2) — Conversion rate, retention, CAC, LTV, ROAS, and NPS, capped off with an all-in-one KPI dashboard function

---
# 🎯 Learning Objective
Today I want to learn:
- [x] How to implement funnel conversion rate, cohort retention, CAC, LTV (3 variants), ROAS, and NPS as reusable functions
- [x] How LTV/CAC ratio and breakeven ROAS turn raw numbers into a "is this business healthy?" verdict
- [x] How to combine all 11 KPIs from both parts into a single `calculate_all_kpis()` dashboard function

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

This part (2 of 2) covers the six KPIs that show up most often in a BA/DA's actual day-to-day work: conversion rate (funnel efficiency), retention rate (product loyalty), CAC (cost to acquire), LTV (value per customer), ROAS (ad efficiency), and NPS (customer satisfaction). Together with Part 1's five statistical foundations, these eleven functions are combined at the end into one dashboard function.

이번 파트(2부 중 2부)는 BA/DA 실무에서 가장 자주 등장하는 6가지 KPI를 다룹니다: 전환율(퍼널 효율), 리텐션율(제품 충성도), CAC(획득 비용), LTV(고객 가치), ROAS(광고 효율), NPS(고객 만족도). 1부의 5가지 통계 기초와 합쳐, 이 11개 함수는 마지막에 하나의 대시보드 함수로 통합됩니다.

### Six core business KPIs / 6가지 핵심 비즈니스 KPI

| KPI | Formula (brief) / 공식(약식) | What it tells you / 알려주는 것 |
|---|---|---|
| CVR (전환율) | conversions / base × 100 | Funnel / marketing efficiency (퍼널·마케팅 효율) |
| Retention (리텐션율) | active on Day N / Day 0 users × 100 | Product loyalty over time (시간에 따른 제품 충성도) |
| CAC | marketing spend / new customers | Cost to acquire one customer (고객 1명 획득 비용) |
| LTV | AOV × freq/yr × lifespan (× margin) | Total value one customer generates (고객 1명의 총 가치) |
| ROAS | ad revenue / ad spend | Ad efficiency, paired with LTV/CAC (광고 효율) |
| NPS | %Promoters − %Detractors | Customer loyalty, −100~+100 (고객 충성도) |

## Why do we use it?
*(When is it useful?)*

Because these six numbers, together, answer the single question every stakeholder actually cares about: "is our growth healthy or are we burning money to get it?" — LTV/CAC ≥ 3x and ROAS above breakeven are the two verdicts a BA is expected to deliver with confidence.

이 6가지 숫자를 합치면 모든 이해관계자가 실제로 궁금해하는 단 하나의 질문에 답할 수 있기 때문입니다: "우리의 성장은 건전한가, 아니면 돈을 태워서 얻고 있는가?" — LTV/CAC ≥ 3배와 손익분기 ROAS 초과 여부는 BA가 자신 있게 내려야 할 두 가지 판정입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

- Diagnosing where in the funnel customers drop off, and by how much.  
  퍼널의 어느 단계에서, 얼마나 이탈하는지 진단할 때.
- Judging whether a product retains users well enough (D1/D7/D30 benchmarks).  
  제품이 사용자를 충분히 붙잡아두는지(D1/D7/D30 기준) 판단할 때.
- Deciding whether a marketing channel is worth its cost (CAC vs LTV, ROAS vs breakeven).  
  마케팅 채널이 비용 대비 가치가 있는지(CAC vs LTV, ROAS vs 손익분기) 판단할 때.
- Tracking customer satisfaction trend over time with NPS.  
  NPS로 시간에 따른 고객 만족도 추세를 추적할 때.

---
# 📝 Syntax

## Basic Syntax

In [ ]:
import numpy as np

revenue = np.array([12000000, 13500000, 11000000])
ad_spend = np.array([4500000, 5200000, 4000000])
visitors = np.array([32000, 35000, 28000])
conversions = np.array([1220, 1370, 1090])

cvr = conversions / visitors * 100         # conversion rate
roas = revenue / ad_spend                  # return on ad spend
print(cvr, roas)

[3.8125     3.91428571 3.89285714] [2.66666667 2.59615385 2.75      ]


## Common Variations

In [ ]:
import numpy as np

# guard against zero denominators, the recurring theme across all business KPIs
spend = np.array([5000000, 0, 4000000])
new_customers = np.array([320, 50, 0])
cac = np.where(new_customers > 0, spend / new_customers, np.nan)
print(cac)

# NPS: three groups from one score array via Boolean Indexing
scores = np.array([9, 10, 7, 3, 8, 9, 5, 10])
promoters = np.sum(scores >= 9) / len(scores) * 100
detractors = np.sum(scores <= 6) / len(scores) * 100
nps = promoters - detractors
print(f"NPS: {nps:+.1f}")

# breakeven ROAS from margin rate -- the profitability threshold
margin_rate = 0.35
breakeven_roas = 1 / margin_rate
print(f"Breakeven ROAS: {breakeven_roas:.2f}x")

[15625.     0.    nan]
NPS: +25.0
Breakeven ROAS: 2.86x


/var/folders/9d/hwl9yp1n3rv1y9j90spl8rmh0000gn/T/ipykernel_8071/3515890130.py:6: RuntimeWarning: divide by zero encountered in divide
  cac = np.where(new_customers > 0, spend / new_customers, np.nan)


---
# 🧪 Small Examples

## Example 1: Conversion Rate — Funnel Drop-off Diagnosis
### 11-6. 전환율 (Conversion Rate, CVR)

$$\text{CVR} = \frac{\text{conversions}}{\text{base}} \times 100 \qquad \text{Overall funnel CVR} = \frac{\text{final conversions}}{\text{initial traffic}} \times 100$$

In [ ]:
import numpy as np

def calc_funnel_cvr(funnel_counts, stage_names=None, label=""):
    """Step-by-step and overall conversion rate through a funnel."""
    counts = np.array(funnel_counts, dtype=float)
    n = len(counts)

    step_cvr = np.zeros(n)
    step_cvr[0] = 100.0
    for i in range(1, n):
        step_cvr[i] = counts[i] / counts[i-1] * 100 if counts[i-1] > 0 else 0

    overall_cvr = counts / counts[0] * 100

    if label:
        names = stage_names or [f"Stage{i+1}" for i in range(n)]
        print(f"[{label}]")
        for i, (name, cnt, scvr, ocvr) in enumerate(zip(names, counts, step_cvr, overall_cvr)):
            drop = 100 - scvr if i > 0 else 0
            print(f"  {name:>12}: {int(cnt):>7,}  step={scvr:5.1f}%  "
                  f"overall={ocvr:5.1f}%  drop={drop:4.1f}%")
        print(f"Final overall conversion: {overall_cvr[-1]:.2f}%")
    return {"step_cvr": step_cvr, "overall_cvr": overall_cvr}

# Business: e-commerce purchase funnel
stages = ["Ad Click", "Landing", "Product View", "Cart", "Purchase"]
counts = [50000, 42000, 28000, 8500, 3200]
calc_funnel_cvr(counts, stage_names=stages, label="E-commerce funnel")

[E-commerce funnel]
      Ad Click:  50,000  step=100.0%  overall=100.0%  drop= 0.0%
       Landing:  42,000  step= 84.0%  overall= 84.0%  drop=16.0%
  Product View:  28,000  step= 66.7%  overall= 56.0%  drop=33.3%
          Cart:   8,500  step= 30.4%  overall= 17.0%  drop=69.6%
      Purchase:   3,200  step= 37.6%  overall=  6.4%  drop=62.4%
Final overall conversion: 6.40%


{'step_cvr': array([100.        ,  84.        ,  66.66666667,  30.35714286,
         37.64705882]),
 'overall_cvr': array([100. ,  84. ,  56. ,  17. ,   6.4])}

## Example 2: Retention Rate — Cohort Analysis
### 11-7. 리텐션율 (Retention Rate)

$$\text{Retention (Day N)} = \frac{\text{users active on Day N}}{\text{total users on signup day}} \times 100$$

In [ ]:
import numpy as np

def calc_retention(cohort_matrix, cohort_labels=None, day_labels=None):
    """Cohort x day retention percentage, using Day 0 as the base (Broadcasting)."""
    matrix = cohort_matrix.astype(float)
    n_cohorts, n_periods = matrix.shape

    base = matrix[:, 0].reshape(-1, 1)   # shape (n_cohorts, 1) -- Broadcasting
    ret_pct = matrix / base * 100

    labels = day_labels or [f"D{i}" for i in range(n_periods)]
    cohorts = cohort_labels or [f"Cohort{i+1}" for i in range(n_cohorts)]
    print(f"{'':>10}" + "".join(f"{l:>8}" for l in labels))
    for coh, row in zip(cohorts, ret_pct):
        print(f"{coh:>10}" + "".join(f"{v:>7.1f}%" for v in row))
    print(f"{'avg':>10}" + "".join(f"{v:>7.1f}%" for v in np.nanmean(ret_pct, axis=0)))
    return ret_pct

# Business: 3 monthly cohorts, D0/D1/D7/D30 retention
base_users = np.array([1500, 1800, 1200])
ret_rates = np.array([
    [1.00, 0.44, 0.26, 0.15],
    [1.00, 0.47, 0.28, 0.17],
    [1.00, 0.41, 0.24, 0.13],
])
# np.round() before astype(int): astype truncates, so 0.41*1200 = 491.9999... -> 491
cohort_matrix = np.round(ret_rates * base_users.reshape(-1, 1)).astype(int)
calc_retention(cohort_matrix, cohort_labels=["Jan", "Feb", "Mar"],
               day_labels=["D0", "D1", "D7", "D30"])

                D0      D1      D7     D30
       Jan  100.0%   44.0%   26.0%   15.0%
       Feb  100.0%   47.0%   28.0%   17.0%
       Mar  100.0%   40.9%   24.0%   13.0%
       avg  100.0%   44.0%   26.0%   15.0%


array([[100.        ,  44.        ,  26.        ,  15.        ],
       [100.        ,  47.        ,  28.        ,  17.        ],
       [100.        ,  40.91666667,  24.        ,  13.        ]])

## Example 3: CAC — Cost per Channel, and the LTV/CAC Health Check
### 11-8. CAC (Customer Acquisition Cost)

$$\text{CAC} = \frac{\text{total marketing/sales spend}}{\text{new customers}}$$

In [ ]:
import numpy as np

def calc_cac(spend, new_customers, labels=None, label=""):
    """Per-period and overall CAC, guarded against zero customers."""
    spend = np.array(spend, dtype=float)
    customers = np.array(new_customers, dtype=float)
    cac_per_period = np.where(customers > 0, spend / customers, np.nan)
    total_cac = np.nansum(spend) / np.nansum(customers)
    if label:
        names = labels or [f"Period{i+1}" for i in range(len(spend))]
        print(f"[{label}]")
        for nm, sp, cu, cac in zip(names, spend, customers, cac_per_period):
            print(f"  {nm:>10}: spend {sp:>10,.0f}  customers {int(cu):>5}  "
                  f"CAC {cac:>9,.0f}")
        print(f"  {'Overall CAC':>10}: {total_cac:>9,.0f}")
    return {"cac_per_period": cac_per_period, "total_cac": total_cac}

# Business: CAC by channel + LTV/CAC health check
channels = ["Search", "SNS", "Email", "Display"]
spend = np.array([5000000, 3000000, 500000, 2000000])
new_cust = np.array([320, 185, 62, 110])
result = calc_cac(spend, new_cust, labels=channels, label="Channel CAC")

# LTV must be MARGIN-adjusted before comparing it to CAC -- CAC is real cash out,
# so the revenue-based LTV would overstate the ratio (see Example 4's margin_ltv).
avg_ltv_revenue = 180000
margin_rate = 0.35
avg_ltv_margin = avg_ltv_revenue * margin_rate
ratio = avg_ltv_margin / result["total_cac"]

print(f"\nRevenue LTV: {avg_ltv_revenue:,.0f}  ->  ratio {avg_ltv_revenue/result['total_cac']:.2f}x (overstated)")
print(f"Margin  LTV: {avg_ltv_margin:,.0f}  ->  ratio {ratio:.2f}x (use this one)")
print(f"LTV/CAC ratio: {ratio:.2f}x  -> "
      f"{'healthy (>=3x)' if ratio >= 3 else 'inefficient (<3x)'}")

[Channel CAC]
      Search: spend  5,000,000  customers   320  CAC    15,625
         SNS: spend  3,000,000  customers   185  CAC    16,216
       Email: spend    500,000  customers    62  CAC     8,065
     Display: spend  2,000,000  customers   110  CAC    18,182
  Overall CAC:    15,510

LTV/CAC ratio: 11.61x  -> healthy (>=3x)


## Example 4: LTV — Simple, Margin & Discounted
### 11-9. LTV (Lifetime Value)

$$\text{Simple LTV} = \text{AOV} \times \text{purchase freq/yr} \times \text{lifespan (yrs)}$$
$$\text{Margin LTV} = \text{Simple LTV} \times \text{margin rate}$$

In [ ]:
import numpy as np

def calc_ltv(aov, freq_per_year, lifespan_years, margin_rate=1.0,
             discount_rate=0.1, label=""):
    """Simple, margin-adjusted, and discounted (DCF) LTV."""
    simple_ltv = aov * freq_per_year * lifespan_years
    margin_ltv = simple_ltv * margin_rate

    annual_profit = aov * freq_per_year * margin_rate
    full_years = int(lifespan_years)
    years = np.arange(1, full_years + 1)
    dcf_ltv = np.sum(annual_profit / (1 + discount_rate) ** years)

    # a fractional lifespan (e.g. 2.5 yrs) must still be counted, pro-rated,
    # or DCF silently covers fewer years than simple/margin LTV
    leftover = lifespan_years - full_years
    if leftover > 0:
        dcf_ltv += annual_profit * leftover / (1 + discount_rate) ** (full_years + 1)

    if label:
        print(f"[{label}] Simple: {simple_ltv:,.0f}  Margin: {margin_ltv:,.0f}  "
              f"DCF: {dcf_ltv:,.0f}")
    return {"simple": simple_ltv, "margin": margin_ltv, "dcf": dcf_ltv}

# Business: LTV by customer segment
segments = {
    "Bronze": {"aov": 35000, "freq": 1.5, "life": 1.0, "margin": 0.25},
    "Silver": {"aov": 85000, "freq": 3.5, "life": 2.5, "margin": 0.32},
    "Gold": {"aov": 180000, "freq": 6.2, "life": 4.0, "margin": 0.38},
    "VIP": {"aov": 450000, "freq": 9.8, "life": 6.5, "margin": 0.42},
}
for seg, p in segments.items():
    calc_ltv(p["aov"], p["freq"], p["life"], p["margin"], label=seg)

[Bronze] Simple: 52,500  Margin: 13,125  DCF: 11,932
[Silver] Simple: 743,750  Margin: 238,000  DCF: 165,223
[Gold] Simple: 4,464,000  Margin: 1,696,320  DCF: 1,344,277
[VIP] Simple: 28,665,000  Margin: 12,039,300  DCF: 8,066,814


## Example 5: ROAS — Channel Efficiency vs Breakeven
### 11-10. ROAS (Return on Ad Spend)

$$\text{ROAS} = \frac{\text{ad-driven revenue}}{\text{ad spend}} \qquad \text{Breakeven ROAS} = \frac{1}{\text{margin rate}}$$

In [ ]:
import numpy as np

def calc_roas(revenue, ad_spend, margin_rate=None, label=""):
    """ROAS, guarded against zero spend, plus a breakeven check."""
    rev = np.array(revenue, dtype=float)
    spend = np.array(ad_spend, dtype=float)
    roas = np.where(spend > 0, rev / spend, np.nan)
    if label:
        print(f"[{label}] avg={np.nanmean(roas):.2f}x  "
              f"max={np.nanmax(roas):.2f}x  min={np.nanmin(roas):.2f}x")
        if margin_rate:
            be = 1 / margin_rate
            print(f"  breakeven ROAS: {be:.2f}x  (margin {margin_rate*100:.0f}%)")
    return roas

# Business: 4 channels x 6 months ROAS, flag campaigns below breakeven
np.random.seed(42)
rev_mat = np.random.randint(800, 3000, (4, 6)) * 10000
spend_mat = np.random.randint(200, 1000, (4, 6)) * 10000
channels = ["Search", "SNS", "Email", "Display"]

roas_mat = calc_roas(rev_mat, spend_mat, margin_rate=0.35, label="All-channel ROAS")

be_roas = 1 / 0.35
below = np.where(roas_mat < be_roas)
print(f"\nCampaigns below breakeven ({be_roas:.2f}x): {len(below[0])}")

[All-channel ROAS] avg=3.66x  max=11.30x  min=1.10x
  breakeven ROAS: 2.86x  (margin 35%)

Campaigns below breakeven (2.86x): 12


## Example 6: NPS — Promoters, Passives, Detractors
### 11-11. NPS 기반 고객 만족도 지수

$$\text{NPS} = \%\text{Promoters} - \%\text{Detractors} \quad (\text{result ranges} -100 \text{ to } +100)$$

In [ ]:
import numpy as np

def calc_nps(scores, label="", breakdown=True):
    """NPS from 0-10 survey scores: 9-10=Promoter, 7-8=Passive, 0-6=Detractor."""
    scores = np.array(scores)
    n = len(scores)
    promoters = np.sum(scores >= 9)
    passives = np.sum((scores >= 7) & (scores <= 8))
    detractors = np.sum(scores <= 6)

    pct_p, pct_pa, pct_d = promoters/n*100, passives/n*100, detractors/n*100
    nps = pct_p - pct_d

    if label:
        print(f"[{label}] n={n:,}")
        if breakdown:
            print(f"  Promoters (9-10): {promoters:>5} ({pct_p:.1f}%)")
            print(f"  Passives  (7-8) : {passives:>5} ({pct_pa:.1f}%)")
            print(f"  Detractors (0-6): {detractors:>5} ({pct_d:.1f}%)")
        grade = "Excellent" if nps >= 50 else "Good" if nps >= 0 else "Poor"
        print(f"  NPS: {nps:+.1f}  [{grade}]")
    return {"nps": nps, "promoters_pct": pct_p, "passives_pct": pct_pa,
            "detractors_pct": pct_d}

# Business: quarterly NPS trend
np.random.seed(42)
quarterly_scores = {
    "Q1": np.random.choice(range(11), 500,
          p=[0.04,0.03,0.04,0.05,0.06,0.10,0.13,0.14,0.18,0.15,0.08]),
    "Q2": np.random.choice(range(11), 600,
          p=[0.03,0.02,0.03,0.04,0.05,0.08,0.12,0.14,0.21,0.19,0.09]),
    "Q3": np.random.choice(range(11), 750,
          p=[0.02,0.02,0.03,0.03,0.04,0.07,0.11,0.13,0.22,0.20,0.13]),
}
q_nps = []
for q, sc in quarterly_scores.items():
    r = calc_nps(sc, breakdown=False, label=q)
    q_nps.append(r["nps"])
print(f"\nQoQ NPS trend: {np.round(q_nps, 1)}")

[Q1] n=500
  NPS: -20.6  [Poor]
[Q2] n=600
  NPS: -11.5  [Poor]
[Q3] n=750
  NPS: +1.9  [Good]

QoQ NPS trend: [-20.6 -11.5   1.9]


## Example 7: Practice — One Mini-Exercise per KPI
### 연습 문제 (Practice)

**Note / 참고:** as in Part 1, the original material has no formal practice-problem subsection for these KPIs — these 6 mini-exercises are self-designed to match this section's own goal of reproducing each function on new numbers.  
1부와 마찬가지로, 원문에는 이 KPI들에 대한 별도의 "연습 문제"가 없습니다 — 아래 6개는 새로운 숫자로 각 함수를 재현한다는 이 섹션의 목표에 맞춰 직접 구성했습니다.

1. **(CVR)** A 3-stage funnel `[10000, 3500, 800]` → compute step and overall CVR.  
   3단계 퍼널 `[10000, 3500, 800]`의 단계별·전체 전환율을 계산하세요.
2. **(Retention)** 2 cohorts, D0/D7/D30: `base=[1000, 1200]`, `rates=[[1.0,0.3,0.15],[1.0,0.35,0.18]]` → print retention %.  
   2개 코호트(D0/D7/D30): `base=[1000, 1200]`, `rates=[[1.0,0.3,0.15],[1.0,0.35,0.18]]`의 리텐션율을 출력하세요.
3. **(CAC)** `spend=[2000000, 1500000]`, `new_customers=[150, 90]` → compute per-channel and total CAC.  
   `spend=[2000000, 1500000]`, `new_customers=[150, 90]`의 채널별·전체 CAC를 계산하세요.
4. **(LTV)** `aov=60000, freq=3, lifespan=2, margin=0.3` → compute simple, margin, and DCF LTV.  
   `aov=60000, freq=3, lifespan=2, margin=0.3`의 simple, margin, DCF LTV를 계산하세요.
5. **(ROAS)** `revenue=[3000000, 1800000]`, `spend=[1200000, 900000]` → compute ROAS, compare to breakeven at `margin=0.3`.  
   `revenue=[3000000, 1800000]`, `spend=[1200000, 900000]`의 ROAS를 계산하고 `margin=0.3` 기준 손익분기와 비교하세요.
6. **(NPS)** `scores=[9,10,8,3,7,9,10,2,6,9]` → compute NPS + grade.
   `scores=[9,10,8,3,7,9,10,2,6,9]`의 NPS와 등급을 계산하세요.

Fill in each `________` below, then run the cell.  
아래 각 `________`를 채운 후 셀을 실행하세요.

In [ ]:
# ✏️ Practice — replace each ________ line below, then run this cell.
# ✏️ 연습 문제 — 아래 각 ________ 줄을 채운 후 셀을 실행하세요.

import numpy as np

print("--- 1: CVR ---")
funnel = np.array([10000, 3500, 800], dtype=float)
step_cvr = np.zeros(len(funnel))
step_cvr[0] = 100.0
for i in range(1, len(funnel)):
    step_cvr[i] = funnel[i] / funnel[i-1] * 100 if funnel[i-1] > 0 else 0
overall_cvr = funnel / funnel[0] * 100
for i, (cnt, s, o) in enumerate(zip(funnel, step_cvr, overall_cvr)):
    print(f"  Stage{i+1}: {int(cnt):>6,}  step={s:5.1f}%  overall={o:5.1f}%  drop={100-s if i else 0:4.1f}%")
print(f"final overall CVR: {overall_cvr[-1]:.2f}%")

print("\n--- 2: Retention ---")
base = np.array([1000, 1200])
rates = np.array([[1.0, 0.3, 0.15], [1.0, 0.35, 0.18]])
matrix = np.round(rates * base.reshape(-1, 1)).astype(int)
ret_pct = matrix / matrix[:, 0].reshape(-1, 1) * 100      # (2,1) so each row divides by its own D0
day_labels = ["D0", "D7", "D30"]
print(f"{'':>8}" + "".join(f"{l:>8}" for l in day_labels))
for i, row in enumerate(ret_pct):
    print(f"{'Cohort'+str(i+1):>8}" + "".join(f"{v:>7.1f}%" for v in row))
print(f"{'avg':>8}" + "".join(f"{v:>7.1f}%" for v in np.nanmean(ret_pct, axis=0)))

print("\n--- 3: CAC ---")
spend = np.array([2000000, 1500000], dtype=float)
new_customers = np.array([150, 90], dtype=float)
cac_per_channel = np.where(new_customers > 0, spend / new_customers, np.nan)
total_cac = np.nansum(spend) / np.nansum(new_customers)
for i, (sp, cu, cac) in enumerate(zip(spend, new_customers, cac_per_channel)):
    print(f"  Channel{i+1}: spend {sp:>10,.0f}  customers {int(cu):>4}  CAC {cac:>8,.0f}")
print(f"  Overall CAC: {total_cac:,.0f}   (NOT the mean of the two: {np.nanmean(cac_per_channel):,.0f})")

print("\n--- 4: LTV ---")
aov, freq, lifespan, margin, discount = 60000, 3, 2, 0.3, 0.1
simple_ltv = aov * freq * lifespan
margin_ltv = simple_ltv * margin
annual_profit = aov * freq * margin
years = np.arange(1, int(lifespan) + 1)
dcf_ltv = np.sum(annual_profit / (1 + discount) ** years)
print(f"  Simple LTV: {simple_ltv:>9,.0f}   (revenue over {lifespan} yrs)")
print(f"  Margin LTV: {margin_ltv:>9,.0f}   (x {margin} margin -- compare THIS to CAC)")
print(f"  DCF LTV   : {dcf_ltv:>9,.0f}   (discounted at {discount:.0%}/yr)")
print(f"  annual profit {annual_profit:,.0f} -> yr1 {annual_profit/1.1:,.0f} + yr2 {annual_profit/1.21:,.0f}")

print("\n--- 5: ROAS ---")
revenue = np.array([3000000, 1800000], dtype=float)
ad_spend = np.array([1200000, 900000], dtype=float)
margin_rate = 0.3
roas = np.where(ad_spend > 0, revenue / ad_spend, np.nan)
breakeven = 1 / margin_rate
print(f"  breakeven ROAS: {breakeven:.2f}x  (margin {margin_rate:.0%})")
for i, x in enumerate(roas):
    verdict = "above breakeven" if x >= breakeven else "BELOW breakeven -> losing money"
    print(f"  Channel{i+1}: {x:.2f}x  -> {verdict}")
print(f"  avg ROAS {np.nanmean(roas):.2f}x -> both channels unprofitable at 30% margin")

print("\n--- 6: NPS ---")
scores = np.array([9, 10, 8, 3, 7, 9, 10, 2, 6, 9])
n = len(scores)
promoters = np.sum(scores >= 9)
passives = np.sum((scores >= 7) & (scores <= 8))
detractors = np.sum(scores <= 6)
pct_p, pct_pa, pct_d = promoters/n*100, passives/n*100, detractors/n*100
nps = pct_p - pct_d
grade = "Excellent" if nps >= 50 else "Good" if nps >= 0 else "Poor"
print(f"  Promoters (9-10): {promoters} ({pct_p:.0f}%)")
print(f"  Passives  (7-8) : {passives} ({pct_pa:.0f}%)   <- excluded from the formula")
print(f"  Detractors (0-6): {detractors} ({pct_d:.0f}%)")
print(f"  NPS: {pct_p:.0f}% - {pct_d:.0f}% = {nps:+.1f}  [{grade}]")
print(f"  n={n} is far too small to act on (see Mistake 2/3)")

print("\n✅ All 6 KPIs computed. Blank the answers out again to re-test yourself later.")
print("✅ 6개 KPI 모두 계산 완료. 나중에 다시 풀어보려면 답을 지우고 실행하세요.")

--- 1: CVR ---

--- 2: Retention ---

--- 3: CAC ---

--- 4: LTV ---

--- 5: ROAS ---

--- 6: NPS ---

✅ Fill in each ________ above with real code, then re-run to see all 6 results.
✅ 위 각 ________ 를 실제 코드로 채운 뒤 다시 실행하면 6개 결과를 모두 볼 수 있습니다.


---
# ⚠️ Common Mistakes

**Mistake 1 — Using inconsistent CVR denominators across reports.**
Visitor-based CVR and click-based CVR are DIFFERENT numbers — mixing them up when comparing reports leads to false conclusions.  
방문자 기준 CVR과 클릭 기준 CVR은 서로 다른 숫자입니다 — 리포트를 비교할 때 이를 혼동하면 잘못된 결론에 이릅니다.  
✅ **Fix:** Always state the denominator explicitly in the metric name or label (e.g. "visitor CVR" vs "click CVR").  
✅ **해결법:** 항상 지표 이름이나 라벨에 분모를 명시하세요(예: "방문자 기준 CVR" vs "클릭 기준 CVR").

**Mistake 2 — Trusting a CVR computed from a tiny sample.**
A 6.5% CVR from only 800 visitors is statistically noisier than a 3.8% CVR from 5,000 visitors — the smaller base can swing wildly with a handful of conversions.  
800명 방문자에서 나온 6.5% CVR은 5,000명에서 나온 3.8% CVR보다 통계적으로 훨씬 불안정합니다 — 표본이 작으면 전환 몇 건만으로도 크게 흔들립니다.  
✅ **Fix:** Treat any CVR from under ~100-200 responses as directional, not conclusive.  
✅ **해결법:** 약 100~200건 미만에서 나온 CVR은 확정적이 아니라 방향성 참고 정도로만 다루세요.

**Mistake 3 — Trusting retention numbers from a cohort that's too small.**
A cohort of 50 users produces a retention rate that can jump 2 percentage points from just one user's behavior — nowhere near stable enough to act on.  
50명짜리 코호트의 리텐션율은 유저 단 1명의 행동만으로도 2%p씩 튈 수 있습니다 — 실무에서 쓰기엔 전혀 안정적이지 않습니다.  
✅ **Fix:** Look for cohorts of 500+ users before treating a retention number as reliable.  
✅ **해결법:** 리텐션 수치를 신뢰하기 전에 코호트 규모가 500명 이상인지 확인하세요.

**Mistake 4 — Comparing retention numbers without agreeing on the definition first.**
"Classic" retention (active on exactly Day N) and "Rolling" retention (active on Day N or any day after) can tell very different stories from the same raw data.  
"Classic" 리텐션(정확히 N일째 활성)과 "Rolling" 리텐션(N일째 이후 아무 때나 활성)은 같은 원본 데이터에서 매우 다른 이야기를 만들 수 있습니다.  
✅ **Fix:** Agree on ONE retention definition across the team before comparing numbers across reports.  
✅ **해결법:** 리포트 간 숫자를 비교하기 전, 팀 내에서 리텐션 정의를 하나로 통일하세요.

**Mistake 5 — Not agreeing on what counts as a "new customer" before computing CAC.**
"New customer" could mean first purchase, app signup, or account creation — each gives a different denominator and therefore a different CAC.  
"신규 고객"은 첫 구매, 앱 가입, 계정 생성 중 무엇이든 의미할 수 있으며, 각각 다른 분모를 주기 때문에 CAC도 달라집니다.  
✅ **Fix:** Define "new customer" explicitly in the function's docstring or the report's methodology note.  
✅ **해결법:** 함수 docstring이나 리포트 방법론 노트에 "신규 고객"을 명시적으로 정의하세요.

**Mistake 6 — Treating the lifespan estimate in LTV as a known fact.**
`avg_customer_lifespan_years` is almost always an ESTIMATE (customers who joined recently haven't finished their lifespan yet) — small changes to this one input swing LTV dramatically.  
`avg_customer_lifespan_years`는 거의 항상 추정치입니다(최근 가입한 고객은 아직 생애주기가 끝나지 않았으므로) — 이 값 하나만 조금 바뀌어도 LTV가 크게 흔들립니다.  
✅ **Fix:** Estimate lifespan conservatively, and show how LTV changes under a pessimistic vs optimistic lifespan assumption.  
✅ **해결법:** 유지 기간을 보수적으로 추정하고, 비관적·낙관적 가정에 따라 LTV가 어떻게 달라지는지 함께 보여주세요.

**Mistake 7 — Comparing ROAS across campaigns measured with different attribution models.**
Last-click, linear, and time-decay attribution can each assign very different revenue to the same campaign — comparing their ROAS values directly isn't apples-to-apples.  
라스트클릭, 선형, 시간감소 어트리뷰션은 같은 캠페인에도 서로 다른 매출을 배분할 수 있습니다 — ROAS 값을 직접 비교하면 공정한 비교가 아닙니다.  
✅ **Fix:** Confirm all campaigns being compared use the SAME attribution model before drawing conclusions.  
✅ **해결법:** 결론을 내리기 전, 비교하는 모든 캠페인이 동일한 어트리뷰션 모델을 쓰는지 확인하세요.

**Mistake 8 — Reading a single NPS snapshot as gospel.**
Response bias (satisfied customers respond more often) skews the absolute NPS number, and blending new-customer NPS with long-time-customer NPS hides real differences between the two groups.  
응답 편향(만족한 고객이 더 자주 응답)은 NPS 절대값을 왜곡하고, 신규 고객과 기존 고객의 NPS를 섞으면 둘 사이의 실제 차이가 가려집니다.  
✅ **Fix:** Track the TREND (quarter over quarter) rather than the absolute number, and segment NPS by customer tenure when possible.  
✅ **해결법:** 절대값보다 추세(분기별)를 추적하고, 가능하다면 고객 가입 기간별로 NPS를 세분화하세요.

---
# 💡 Tips
Useful tips or shortcuts

- LTV/CAC ≥ 3x and ROAS ≥ breakeven (`1/margin`) are the two numbers a stakeholder will ask for first — always have both ready.  
  LTV/CAC ≥ 3배와 ROAS ≥ 손익분기(`1/마진`)는 이해관계자가 가장 먼저 물어볼 두 숫자입니다 — 항상 둘 다 준비해두세요.
- When comparing NPS or retention across cohorts, show the TREND (quarter over quarter) rather than a single snapshot — a single number is easy to misread.  
  코호트 간 NPS나 리텐션을 비교할 때는 단일 스냅샷보다 추세(분기별)를 함께 보여주세요 — 숫자 하나는 오독하기 쉽습니다.
- Every business KPI function in this section shares one defensive pattern: `np.where(denominator > 0, ratio, np.nan)` — write it once, reuse it everywhere.  
  이번 섹션의 모든 비즈니스 KPI 함수는 하나의 방어 패턴을 공유합니다: `np.where(분모 > 0, 비율, np.nan)` — 한 번 작성해서 어디서나 재사용하세요.

---
# 🔗 Related Concepts

```
Section 11 (Part 1/2) — KPI Foundations
   (mean, median, std/var, growth rate, change rate)
        ↓
🔵 Section 11 (Part 2/2) — Core Business KPIs   ← you are here
   (conversion rate, retention, CAC, LTV, ROAS, NPS)
        ↓
Section 12 — Missing Value Handling
        ↓
Pandas → SQL → Tableau
```

*How is today's topic connected to other concepts?*

All 11 KPIs from both parts converge in this notebook's capstone function — `calculate_all_kpis()` calls mean, median, std, growth rate, and change rate from Part 1 alongside this part's CVR, CAC, LTV, ROAS, and NPS, wrapped into one dashboard-style report. This is the natural end point of the entire NumPy series: every tool from Sections 1-10 was building toward being able to write exactly this kind of function.

두 파트의 11개 KPI 전부가 이 노트북의 캡스톤 함수에서 합쳐집니다 — `calculate_all_kpis()`는 1부의 평균·중앙값·표준편차·성장률·증감률을 이번 파트의 CVR·CAC·LTV·ROAS·NPS와 함께 호출해서 하나의 대시보드 스타일 리포트로 감쌉니다. 이는 전체 NumPy 시리즈의 자연스러운 종착점입니다 — 섹션 1~10의 모든 도구는 정확히 이런 함수를 작성할 수 있게 되는 것을 목표로 쌓아온 것입니다.

---
# 💼 Business Example — The Capstone: `calculate_all_kpis()`
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
Your manager wants ONE function that takes a period's raw numbers (revenue, cost, visitors, conversions, new customers, optionally NPS survey scores and LTV inputs) and prints a complete KPI dashboard — combining every function built across both parts of this section.  
매니저가 한 기간의 원시 숫자(매출, 비용, 방문자, 전환, 신규 고객, 선택적으로 NPS 설문 응답과 LTV 입력값)를 받아서 완전한 KPI 대시보드를 출력하는 함수 하나를 원합니다 — 이 섹션의 두 파트에서 만든 모든 함수를 하나로 결합합니다.

**To-do / 할 일:**
- [x] Combine mean/median/std/growth/change (Part 1) with CVR/CAC/LTV/ROAS/NPS (Part 2) into one function.  
      평균/중앙값/표준편차/성장률/증감률(1부)과 CVR/CAC/LTV/ROAS/NPS(2부)를 하나의 함수로 결합합니다.
- [x] Guard every ratio against a zero denominator.  
      모든 비율 계산에서 분모 0을 방어합니다.
- [x] Print a readable, labeled dashboard, not just raw numbers.  
      단순 숫자가 아니라 읽기 쉽고 라벨이 붙은 대시보드를 출력합니다.

In [ ]:
import numpy as np

def calculate_all_kpis(
    revenue_arr, cost_arr, visitors_arr, conversions_arr, new_customers_arr,
    nps_scores=None, avg_order_value=None, purchase_freq=None,
    customer_lifespan=None, margin_rate=0.35, period_labels=None, verbose=True
):
    """
    Calculate all 11 KPIs from Section 11 (both parts) in one call.
    Returns a dict containing every KPI value.
    """
    r = np.array(revenue_arr, dtype=float)
    c = np.array(cost_arr, dtype=float)
    v = np.array(visitors_arr, dtype=float)
    n = np.array(conversions_arr, dtype=float)
    nc = np.array(new_customers_arr, dtype=float)

    kpis = {}

    # 1. Mean
    kpis["mean_revenue"] = np.mean(r)
    kpis["mean_cost"] = np.mean(c)

    # 2. Median
    kpis["median_revenue"] = np.median(r)

    # 3. Std Dev / CV
    kpis["std_revenue"] = np.std(r, ddof=1)
    kpis["cv_revenue"] = kpis["std_revenue"] / kpis["mean_revenue"] * 100

    # 4. Growth Rate (period over period)
    growth = np.full(len(r), np.nan)
    valid = r[:-1] != 0                   # guard the denominator, like every other ratio here
    growth[1:][valid] = (r[1:][valid] - r[:-1][valid]) / r[:-1][valid] * 100
    kpis["mom_growth"] = growth
    kpis["avg_mom_growth"] = float(np.nanmean(growth))

    # 5. Change Rate (latest period vs previous)
    kpis["latest_change_abs"] = float(r[-1] - r[-2]) if len(r) >= 2 else np.nan
    kpis["latest_change_pct"] = (
        float((r[-1] - r[-2]) / r[-2] * 100) if len(r) >= 2 else np.nan
    )

    # 6. Conversion Rate
    cvr = np.where(v > 0, n / v * 100, np.nan)
    kpis["cvr_by_period"] = cvr
    kpis["avg_cvr"] = float(np.nanmean(cvr))

    # 7. Retention -- simplified here as overall conversion (a true cohort
    # retention needs a 2D cohort matrix; see Example 2 for the full version)
    kpis["overall_cvr"] = float(np.sum(n) / np.sum(v) * 100)

    # 8. CAC
    cac_by_period = np.where(nc > 0, c / nc, np.nan)
    kpis["cac_by_period"] = cac_by_period
    kpis["avg_cac"] = float(np.sum(c) / np.sum(nc)) if np.sum(nc) > 0 else np.nan

    # 9. LTV
    if all(x is not None for x in [avg_order_value, purchase_freq, customer_lifespan]):
        simple_ltv = avg_order_value * purchase_freq * customer_lifespan
        margin_ltv = simple_ltv * margin_rate
        kpis["simple_ltv"] = simple_ltv
        kpis["margin_ltv"] = margin_ltv
        kpis["ltv_cac_ratio"] = margin_ltv / kpis["avg_cac"] if kpis["avg_cac"] else np.nan
    else:
        kpis["simple_ltv"] = kpis["margin_ltv"] = kpis["ltv_cac_ratio"] = np.nan

    # 10. ROAS
    roas = np.where(c > 0, r / c, np.nan)
    kpis["roas_by_period"] = roas
    kpis["avg_roas"] = float(np.nanmean(roas))
    kpis["breakeven_roas"] = 1 / margin_rate if margin_rate > 0 else np.nan

    # 11. NPS
    if nps_scores is not None:
        sc = np.array(nps_scores)
        p = np.sum(sc >= 9) / len(sc) * 100
        d = np.sum(sc <= 6) / len(sc) * 100
        kpis["nps"] = float(p - d)
    else:
        kpis["nps"] = np.nan

    if verbose:
        labels = period_labels or [f"P{i+1}" for i in range(len(r))]
        be_r = kpis["breakeven_roas"]

        print("=" * 60)
        print("   KPI Dashboard")
        print("=" * 60)
        print("\n  Basic stats")
        print(f"    Mean revenue      : {kpis['mean_revenue']:>12,.0f}")
        print(f"    Median revenue    : {kpis['median_revenue']:>12,.0f}")
        print(f"    Std dev           : {kpis['std_revenue']:>12,.0f}")
        print(f"    CV                : {kpis['cv_revenue']:>11.1f}%")

        print("\n  Growth")
        print(f"    Avg period growth : {kpis['avg_mom_growth']:>+11.2f}%")
        print(f"    Latest change     : {kpis['latest_change_pct']:>+11.2f}%  "
              f"({kpis['latest_change_abs']:>+,.0f})")

        print("\n  Efficiency")
        print(f"    Avg CVR           : {kpis['avg_cvr']:>11.2f}%")
        print(f"    Avg CAC           : {kpis['avg_cac']:>12,.0f}")
        print(f"    Avg ROAS          : {kpis['avg_roas']:>11.2f}x   "
              f"(breakeven {be_r:.2f}x)")
        roas_ok = "profitable" if kpis['avg_roas'] >= be_r else "unprofitable"
        print(f"    ROAS status       : {roas_ok}")

        print("\n  Customer value")
        if not np.isnan(kpis["margin_ltv"]):
            print(f"    Margin LTV        : {kpis['margin_ltv']:>12,.0f}")
            lc = kpis['ltv_cac_ratio']
            ltv_ok = "healthy" if lc >= 3 else "needs improvement"
            print(f"    LTV/CAC ratio     : {lc:>11.2f}x   ({ltv_ok})")
        if not np.isnan(kpis["nps"]):
            nps_grade = ("Excellent" if kpis["nps"] >= 50
                         else "Good" if kpis["nps"] >= 0 else "Poor")
            print(f"    NPS               : {kpis['nps']:>+11.1f}   [{nps_grade}]")

        print("\n  Period detail")
        print(f"  {'Period':>6} {'Revenue':>10} {'ROAS':>8} {'CVR%':>7} "
              f"{'CAC':>10} {'Growth%':>8}")
        print("  " + "-" * 52)
        for i, lbl in enumerate(labels):
            mom_str = (f"{kpis['mom_growth'][i]:>+7.1f}%"
                       if not np.isnan(kpis['mom_growth'][i]) else f"{'--':>8}")
            roas_str = (f"{kpis['roas_by_period'][i]:>7.2f}x"
                        if not np.isnan(kpis['roas_by_period'][i]) else f"{'--':>8}")
            cvr_str = (f"{kpis['cvr_by_period'][i]:>6.2f}%"
                       if not np.isnan(kpis['cvr_by_period'][i]) else f"{'--':>7}")
            cac_str = (f"{kpis['cac_by_period'][i]:>9,.0f}"
                       if not np.isnan(kpis['cac_by_period'][i]) else f"{'--':>10}")
            print(f"  {lbl:>6} {r[i]:>10,.0f} {roas_str} {cvr_str} {cac_str} {mom_str}")

        print("=" * 60)

    return kpis


# ── Demo: 6 months of data across every KPI at once ──────────────────────
np.random.seed(42)
months = ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]

revenue = np.array([12000000, 13500000, 11000000, 14000000, 16000000, 15000000])
ad_spend = np.array([4500000, 5200000, 4000000, 5500000, 6000000, 5800000])
visitors = np.array([32000, 35000, 28000, 37000, 41000, 39000])
conversions = np.array([1220, 1370, 1090, 1410, 1600, 1520])
new_customers = np.array([420, 490, 380, 520, 580, 550])
nps_scores = np.random.choice(
    range(11), 800,
    p=[0.02,0.02,0.03,0.04,0.05,0.08,0.12,0.14,0.20,0.18,0.12]
)

result = calculate_all_kpis(
    revenue_arr=revenue, cost_arr=ad_spend, visitors_arr=visitors,
    conversions_arr=conversions, new_customers_arr=new_customers,
    nps_scores=nps_scores, avg_order_value=85000, purchase_freq=4.2,
    customer_lifespan=3.5, margin_rate=0.35, period_labels=months, verbose=True
)

   KPI Dashboard

  Basic stats
    Mean revenue      :   13,583,333
    Median revenue    :   13,750,000
    Std dev           :    1,855,173
    CV                :        13.7%

  Growth
    Avg period growth :       +5.86%
    Latest change     :       -6.25%  (-1,000,000)

  Efficiency
    Avg CVR           :        3.87%
    Avg CAC           :       10,544
    Avg ROAS          :        2.64x   (breakeven 2.86x)
    ROAS status       : unprofitable

  Customer value
    Margin LTV        :      437,325
    LTV/CAC ratio     :       41.48x   (healthy)
    NPS               :        -7.2   [Poor]

  Period detail
  Period    Revenue     ROAS    CVR%        CAC  Growth%
  ----------------------------------------------------
     Jan 12,000,000    2.67x   3.81%    10,714       --
     Feb 13,500,000    2.60x   3.91%    10,612   +12.5%
     Mar 11,000,000    2.75x   3.89%    10,526   -18.5%
     Apr 14,000,000    2.55x   3.81%    10,577   +27.3%
     May 16,000,000    2.67x   3.90%  

---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

This part builds six core business KPIs as reusable functions: conversion rate tracks funnel efficiency (with each drop-off pinpointed), retention rate tracks product loyalty across cohorts using row-wise Broadcasting, CAC and LTV together answer whether a business is acquiring customers profitably (LTV/CAC ≥ 3x is the health threshold), ROAS measures ad efficiency against a breakeven point derived from margin rate, and NPS summarizes customer satisfaction as a single -100-to-+100 score. Every function shares one defensive habit — guarding divisions with `np.where(denominator > 0, ...)` — and all eleven functions from both parts of this section converge in `calculate_all_kpis()`, a single call that prints a full KPI dashboard from raw period data.

이 파트는 6가지 핵심 비즈니스 KPI를 재사용 가능한 함수로 만듭니다: 전환율은 퍼널 효율을 단계별 이탈까지 짚어가며 추적하고, 리텐션율은 행 방향 Broadcasting으로 코호트별 제품 충성도를 추적하며, CAC와 LTV는 함께 비즈니스가 수익성 있게 고객을 확보하고 있는지 답합니다(LTV/CAC ≥ 3배가 건전성 기준). ROAS는 마진율에서 도출된 손익분기점 대비 광고 효율을 측정하고, NPS는 고객 만족도를 −100~+100 사이의 단일 점수로 요약합니다. 모든 함수는 하나의 방어 습관을 공유합니다 — `np.where(분모 > 0, ...)`로 나눗셈을 방어하는 것 — 그리고 이 섹션 두 파트의 11개 함수 전부가 `calculate_all_kpis()`에서 합쳐져, 원시 기간 데이터로부터 전체 KPI 대시보드를 출력하는 단일 호출이 됩니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Conversion rate, retention, CAC, LTV, ROAS, and NPS are six reusable, divide-by-zero-safe functions that together answer "is this business healthy?" — and combined with Part 1's five statistics, they form the `calculate_all_kpis()` dashboard that closes out this section.

> 전환율, 리텐션율, CAC, LTV, ROAS, NPS는 함께 "이 비즈니스는 건전한가?"에 답하는 6가지 재사용 가능하고 0나눗셈에 안전한 함수이며, 1부의 5가지 통계와 합쳐져 이 섹션을 마무리 하는 `calculate_all_kpis()` 대시보드가 됩니다.

---
# ❓ Review Questions

**Q1.** In a purchase funnel, what's the difference between "step CVR" and "overall CVR," and why might a stage have a low step CVR but the funnel still perform well overall?
구매 퍼널에서 "단계 CVR"과 "전체 CVR"의 차이는 무엇이며, 왜 한 단계의 CVR이 낮아도 전체 퍼널 성과는 괜찮을 수 있나요?

**Q2.** Why does `calc_retention()` use `matrix[:, 0].reshape(-1, 1)` instead of just `matrix[:, 0]` when dividing?
`calc_retention()`은 나눗셈할 때 왜 `matrix[:, 0]`이 아니라 `matrix[:, 0].reshape(-1, 1)`을 사용하나요?

**Q3.** What does an LTV/CAC ratio of 1.5x tell you about a business, and why is that different from a ratio of 3.5x?
LTV/CAC 비율이 1.5배라는 것은 비즈니스에 대해 무엇을 말해주며, 3.5배와는 왜 다른가요?

**Q4.** How is "breakeven ROAS" derived from margin rate, and what does it mean if a channel's actual ROAS is below it?
"손익분기 ROAS"는 마진율로부터 어떻게 도출되며, 채널의 실제 ROAS가 그보다 낮다면 무엇을 의미하나요?

**Q5.** In `calc_nps()`, why are scores of 7-8 ("Passives") excluded from the NPS formula entirely?
`calc_nps()`에서 7~8점("Passives")은 왜 NPS 공식에서 완전히 제외되나요?

---
*📅 Try answering these again in a few days.*